In [5]:
%load_ext autoreload
%autoreload 2

## Check for Proper Installation

In [6]:
import sys
print(sys.executable)

import subprocess
subprocess.run(["pip", "show", "chromadb"], capture_output=True, text=True).stdout

sys.path.append("RAG") # temporarily append RAG submodule for working imports

c:\Users\Moirai\anaconda3\envs\Hermes\python.exe


## Create Chroma Store

In [7]:
from RAG.rag.chroma import ChromaStore
from RAG.rag.indexer import Indexer

store = ChromaStore()
indexer = Indexer(store)
print("Collections loaded:", list(store.collections.keys()))

Collections loaded: ['quests', 'lore']


## Load JSON Data

In [8]:
import json

data = {}
with open ("./data/json/data.json") as f:
    data = json.load(f)

## Test If JSON Data was Loaded Correctly

In [9]:
print(f"Total Number of Quests: {len(data)}")
print("\nTitles of a few quests:\n")

i = 0
for title, content in data.items():
    print(title)
    if i > 5:
        break
    i += 1

Total Number of Quests: 147

Titles of a few quests:

Escape the Nautiloid
Rescue the Illithids' Captive
Find a Cure
Find the Nightsong
Gather Your Allies
Travel to Moonrise Towers
Daughter of Darkness


## Create a List of Dictionaries

In [10]:
data_to_store = []

i = 0
for title, content in data.items():
    new_item = {}
    new_item["id"] = f"q_{str(i).zfill(4)}"
    new_item["text"] = title
    # print(content["Walkthrough"])
    if len(content["Walkthrough"]) != 0:
        new_item["metadata"] = {"Walkthrough": content["Walkthrough"]}
        data_to_store.append(new_item)
    else:
        print(f"Skipping due to invalid walkthrough. Length of walkthrough: {len(content["Walkthrough"])}")
    i += 1

print(data_to_store[1])

Skipping due to invalid walkthrough. Length of walkthrough: 0
Skipping due to invalid walkthrough. Length of walkthrough: 0
Skipping due to invalid walkthrough. Length of walkthrough: 0
{'id': 'q_0001', 'text': "Rescue the Illithids' Captive", 'metadata': {'Walkthrough': "After the first imp fight on the Nautiloid, the party must enter a room with two unconscious victims of the mind flayers, a high elf and human. Towards the north-east side of the room,  Shadowheart is trapped in a mind flayer pod.\nIf speaking to Shadowheart, she suggests trying to open the pod by using a nearby panel. The party can try one of the following options to physically free her from the pod:\nGrip the pod's lid and pull.\n[BERSERKER][STRENGTH]RIP THE POD OPEN.(DC 10)\n[BARBARIAN][STRENGTH]Don't give up. Tear this thing open.(DC 10)\nOtherwise, the pod does not budge.\nIf examining the nearby console, an empty socket is revealed. Sorcerers, Warlocks, and Wizards can attempt to study it and then inscribe the r

## Add Data to Vector Database

In [11]:
import sys
# Hack needed to import RAG from the project's root
sys.path.append("RAG")

from rag.rag_api import RAGAPI

myrag = RAGAPI()
# myrag.add_bulk("quests", data_to_store)

## Query Database for Walkthroughs

In [12]:
# This allows you to search the database by the title of the objective
# The title does not need to be exact
results = myrag.query("The mission with the wizard")

print(f"Number of results: {len(results)}")

if len(results) > 0:
    print(f"Displaying one result, \"{results[0]["text"]}\", as a sample:\n\n")
    print(results[0]["metadata"]["Walkthrough"])

#for r in results:
#    print(r["metadata"]["Walkthrough"])

Number of results: 3
Displaying one result, "The Wizard of Waterdeep", as a sample:


Gale’s story spans the entire game. This walkthrough contains spoilers for all three Acts.
Recruiting Gale
Gale can be recruited after the Prologue is completed and the party has washed up on the Ravaged Beach. Gale is northeast of the Nautiloid wreckage, trapped in a purple portal - an Ancient Sigil Circle X:224Y:326. He pops his hand out of the portal, asking for help. If the party want Gale as a member, it is recommended to grab his hand and pull with a  DC15 Strength Check to free him from the trap. Warlocks and Sorcerers can use a Charisma check to attune with the sigil’s magic and pull Gale, while Rogues can pass a  DC15 Dexterity Check to similar ends. A Dark Urge origin has a unique option which, if selected, prevents Gale from joining the party.
If freed from the portal he introduces himself as Gale of Waterdeep. He recognizes the party as fellow captives of the Nautiloid and suggests they tr